# Adani Green — 4H Candles with EMA 50

Hourly Adani Green candles resampled into **4-hour** candles (09:15–13:15 and 13:15–15:30 each session), with a **50 EMA of close** and the **EMA 50 angle**.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root whether Jupyter starts there or in a notebook folder."""
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate the visualizer project root.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from visualizer import load_candle_json

pd.options.display.float_format = "{:,.2f}".format

## Load Adani Green hourly data

In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "adanigreen_2023-12-01_2026-08-19.json"

hourly = load_candle_json(DATA_PATH)
required_columns = {"timestamp", "open", "high", "low", "close", "volume"}
missing_columns = required_columns.difference(hourly.columns)
if missing_columns:
    raise ValueError(f"Missing required column(s): {sorted(missing_columns)}")

numeric_columns = ["open", "high", "low", "close", "volume"]
hourly[numeric_columns] = hourly[numeric_columns].apply(pd.to_numeric, errors="coerce")
hourly = (
    hourly.dropna(subset=["timestamp", "open", "high", "low", "close"])
    .sort_values("timestamp")
    .drop_duplicates("timestamp", keep="last")
    .set_index("timestamp")
)

print(f"Loaded {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"{len(hourly):,} hourly candles from {hourly.index.min()} to {hourly.index.max()}")
display(hourly.head())

Loaded data/raw/adanigreen_2023-12-01_2026-08-19.json
4,698 hourly candles from 2023-12-01 09:15:00+05:30 to 2026-08-19 15:15:00+05:30


,open,high,low,close,volume,open_interest
timestamp,,,,,,
2023-12-01 09:15:00+05:30,"1,042.00","1,045.00","1,022.05","1,028.35",382504,0
2023-12-01 10:15:00+05:30,"1,029.00","1,044.85","1,026.10","1,033.05",314003,0
2023-12-01 11:15:00+05:30,"1,033.05","1,035.45","1,030.05","1,032.50",70226,0
2023-12-01 12:15:00+05:30,"1,032.50","1,036.40","1,028.50","1,034.45",69477,0
2023-12-01 13:15:00+05:30,"1,034.95","1,035.80","1,030.25","1,033.85",62662,0


## Resample to 4H candles

Buckets are anchored at the 09:15 market open, so each session gives two candles: 09:15 (four hourly bars) and 13:15 (the remaining bars to the close).

In [3]:
price = (
    hourly[numeric_columns]
    .resample("4h", origin="start_day", offset="9h15min", label="left", closed="left")
    .agg({"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"})
    .dropna(subset=["open", "high", "low", "close"])
)

print(f"{len(price):,} 4H candles from {price.index.min()} to {price.index.max()}")
display(price.head())

1,344 4H candles from 2023-12-01 09:15:00+05:30 to 2026-08-19 13:15:00+05:30


,open,high,low,close,volume
timestamp,,,,,
2023-12-01 09:15:00+05:30,"1,042.00","1,045.00","1,022.05","1,034.45",836210
2023-12-01 13:15:00+05:30,"1,034.95","1,037.50","1,020.00","1,021.95",321675
2023-12-04 09:15:00+05:30,"1,160.00","1,184.00","1,092.55","1,114.30",4195391
2023-12-04 13:15:00+05:30,"1,114.30","1,133.00","1,108.90","1,124.85",1633006
2023-12-05 09:15:00+05:30,"1,126.00","1,348.50","1,096.10","1,348.50",16443709


## Calculate EMA 50 and EMA 50 angle

In [4]:
EMA_PERIOD = 50
ANGLE_LOOKBACK = 3


def angle(series: pd.Series, window: int = 3) -> pd.Series:
    """Return the line angle in degrees using average price-point slope per bar."""
    numeric = pd.to_numeric(series, errors="coerce").astype(float)
    slope = (numeric - numeric.shift(window)) / float(window)
    return pd.Series(
        np.degrees(np.arctan(np.clip(slope, -10, 10))),
        index=series.index,
    )


EMA_COL = f"ema_{EMA_PERIOD}"
ANGLE_COL = f"ema_{EMA_PERIOD}_angle"

price[EMA_COL] = price["close"].ewm(span=EMA_PERIOD, adjust=False, min_periods=EMA_PERIOD).mean()
price[ANGLE_COL] = angle(price[EMA_COL], ANGLE_LOOKBACK)

latest = price.iloc[-1]
print(
    f"Latest close: ₹{latest['close']:,.2f} | "
    f"EMA {EMA_PERIOD}: ₹{latest[EMA_COL]:,.2f} | "
    f"EMA {EMA_PERIOD} angle: {latest[ANGLE_COL]:.2f}°"
)

Latest close: ₹1,293.00 | EMA 50: ₹1,383.79 | EMA 50 angle: -74.93°


## 4H data frame

In [5]:
price_4h = price.rename_axis("timestamp").reset_index()
price_4h

,timestamp,open,high,low,close,volume,ema_50,ema_50_angle
0,2023-12-01 09:15:00+05:30,"1,042.00","1,045.00","1,022.05","1,034.45",836210,NaN,NaN
1,2023-12-01 13:15:00+05:30,"1,034.95","1,037.50","1,020.00","1,021.95",321675,NaN,NaN
2,2023-12-04 09:15:00+05:30,"1,160.00","1,184.00","1,092.55","1,114.30",4195391,NaN,NaN
3,2023-12-04 13:15:00+05:30,"1,114.30","1,133.00","1,108.90","1,124.85",1633006,NaN,NaN
4,2023-12-05 09:15:00+05:30,"1,126.00","1,348.50","1,096.10","1,348.50",16443709,NaN,NaN
...,...,...,...,...,...,...,...,...
1339,2026-08-17 13:15:00+05:30,"1,332.90","1,333.50","1,324.10","1,327.60",247313,"1,397.95",-69.34
1340,2026-08-18 09:15:00+05:30,"1,327.60","1,336.40","1,316.00","1,320.90",632228,"1,394.93",-70.99
1341,2026-08-18 13:15:00+05:30,"1,320.90","1,321.80","1,308.00","1,308.00",292425,"1,391.52",-72.13
1342,2026-08-19 09:15:00+05:30,"1,312.90","1,312.90","1,281.10","1,288.80",1004075,"1,387.49",-74.00


## Price chart

4H candlesticks with EMA 50 on top, the EMA 50 angle (three-bar price-point slope, in degrees) in the middle, and volume at the bottom.

In [6]:
# Use None for the full history, or set a number such as 300 for recent candles only.
PLOT_BARS = None
plot_df = price.copy() if PLOT_BARS is None else price.tail(PLOT_BARS).copy()

volume_colors = plot_df["close"].ge(plot_df["open"]).map(
    {True: "rgba(22, 163, 74, 0.42)", False: "rgba(220, 38, 38, 0.42)"}
)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.68, 0.18, 0.14],
    vertical_spacing=0.04,
)

fig.add_trace(
    go.Candlestick(
        x=plot_df.index,
        open=plot_df["open"],
        high=plot_df["high"],
        low=plot_df["low"],
        close=plot_df["close"],
        name="ADANIGREEN 4H",
        increasing_line_color="#16a34a",
        decreasing_line_color="#dc2626",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=plot_df[EMA_COL],
        mode="lines",
        name=f"EMA {EMA_PERIOD}",
        line=dict(color="#2563eb", width=2.0),
        hovertemplate=f"EMA {EMA_PERIOD}: ₹%{{y:,.2f}}<extra></extra>",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=plot_df[ANGLE_COL],
        mode="lines",
        name=f"EMA {EMA_PERIOD} angle",
        line=dict(color="#2563eb", width=1.6),
        hovertemplate=f"EMA {EMA_PERIOD} angle: %{{y:.2f}}°<extra></extra>",
    ),
    row=2,
    col=1,
)
fig.add_hline(y=0, line=dict(color="#94a3b8", width=1, dash="dot"), row=2, col=1)
fig.add_trace(
    go.Bar(
        x=plot_df.index,
        y=plot_df["volume"],
        name="Volume",
        marker_color=volume_colors,
        hovertemplate="Volume: %{y:,.0f}<extra></extra>",
    ),
    row=3,
    col=1,
)

fig.update_layout(
    title=dict(text=f"ADANIGREEN 4H Price — EMA {EMA_PERIOD} and EMA {EMA_PERIOD} Angle", x=0.01),
    height=950,
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=70, r=30, t=100, b=50),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    xaxis_rangeslider_visible=False,
)
# Hide weekends and overnight gaps so 4H candles sit next to each other.
fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
        dict(bounds=[15.5, 9.25], pattern="hour"),
    ]
)
fig.update_yaxes(title_text="Price (₹)", row=1, col=1)
fig.update_yaxes(title_text="Angle (°)", row=2, col=1)
fig.update_yaxes(title_text="Volume", row=3, col=1)
fig.show()

## 1H candles with a Donchian 20 channel

The hourly candles keep their own frame. The Donchian channel uses the highest high and lowest low of the last **20** hourly candles, with the midline halfway between them. Entry and exit markers get added later.

In [7]:
DONCHIAN_PERIOD = 20

intraday = hourly[numeric_columns].copy()
intraday["donchian_upper"] = intraday["high"].rolling(DONCHIAN_PERIOD, min_periods=DONCHIAN_PERIOD).max()
intraday["donchian_lower"] = intraday["low"].rolling(DONCHIAN_PERIOD, min_periods=DONCHIAN_PERIOD).min()
intraday["donchian_mid"] = (intraday["donchian_upper"] + intraday["donchian_lower"]) / 2
intraday["donchian_width"] = intraday["donchian_upper"] - intraday["donchian_lower"]

latest_1h = intraday.iloc[-1]
print(
    f"{len(intraday):,} hourly candles | "
    f"Latest close: ₹{latest_1h['close']:,.2f} | "
    f"Donchian {DONCHIAN_PERIOD}: ₹{latest_1h['donchian_lower']:,.2f} – ₹{latest_1h['donchian_upper']:,.2f}"
)

4,698 hourly candles | Latest close: ₹1,293.00 | Donchian 20: ₹1,281.10 – ₹1,336.50


### 1H data frame

In [8]:
price_1h = intraday.rename_axis("timestamp").reset_index()
price_1h

,timestamp,open,high,low,close,volume,donchian_upper,donchian_lower,donchian_mid,donchian_width
0,2023-12-01 09:15:00+05:30,"1,042.00","1,045.00","1,022.05","1,028.35",382504,NaN,NaN,NaN,NaN
1,2023-12-01 10:15:00+05:30,"1,029.00","1,044.85","1,026.10","1,033.05",314003,NaN,NaN,NaN,NaN
2,2023-12-01 11:15:00+05:30,"1,033.05","1,035.45","1,030.05","1,032.50",70226,NaN,NaN,NaN,NaN
3,2023-12-01 12:15:00+05:30,"1,032.50","1,036.40","1,028.50","1,034.45",69477,NaN,NaN,NaN,NaN
4,2023-12-01 13:15:00+05:30,"1,034.95","1,035.80","1,030.25","1,033.85",62662,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
4693,2026-08-19 11:15:00+05:30,"1,289.50","1,293.80","1,287.60","1,292.90",61921,"1,348.00","1,281.10","1,314.55",66.90
4694,2026-08-19 12:15:00+05:30,"1,293.00","1,293.80","1,288.00","1,288.80",61683,"1,348.00","1,281.10","1,314.55",66.90
4695,2026-08-19 13:15:00+05:30,"1,288.00","1,292.50","1,286.50","1,291.20",77575,"1,348.00","1,281.10","1,314.55",66.90
4696,2026-08-19 14:15:00+05:30,"1,291.60","1,297.80","1,287.60","1,290.00",245370,"1,346.90","1,281.10","1,314.00",65.80


### 1H Donchian chart

Hourly candlesticks inside the shaded Donchian 20 channel, with the midline dashed and volume below.

In [9]:
# Use None for the full history, or set a number such as 500 for recent candles only.
PLOT_BARS_1H = 500
plot_1h = intraday.copy() if PLOT_BARS_1H is None else intraday.tail(PLOT_BARS_1H).copy()

volume_colors_1h = plot_1h["close"].ge(plot_1h["open"]).map(
    {True: "rgba(22, 163, 74, 0.42)", False: "rgba(220, 38, 38, 0.42)"}
)

fig_1h = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.78, 0.22],
    vertical_spacing=0.04,
)

fig_1h.add_trace(
    go.Scatter(
        x=plot_1h.index,
        y=plot_1h["donchian_upper"],
        mode="lines",
        name=f"Donchian {DONCHIAN_PERIOD} upper",
        line=dict(color="#0ea5e9", width=1.6),
        hovertemplate="Upper: ₹%{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)
fig_1h.add_trace(
    go.Scatter(
        x=plot_1h.index,
        y=plot_1h["donchian_lower"],
        mode="lines",
        name=f"Donchian {DONCHIAN_PERIOD} lower",
        line=dict(color="#0ea5e9", width=1.6),
        fill="tonexty",
        fillcolor="rgba(14, 165, 233, 0.12)",
        hovertemplate="Lower: ₹%{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)
fig_1h.add_trace(
    go.Scatter(
        x=plot_1h.index,
        y=plot_1h["donchian_mid"],
        mode="lines",
        name=f"Donchian {DONCHIAN_PERIOD} mid",
        line=dict(color="#f59e0b", width=1.4, dash="dash"),
        hovertemplate="Mid: ₹%{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)
fig_1h.add_trace(
    go.Candlestick(
        x=plot_1h.index,
        open=plot_1h["open"],
        high=plot_1h["high"],
        low=plot_1h["low"],
        close=plot_1h["close"],
        name="ADANIGREEN 1H",
        increasing_line_color="#16a34a",
        decreasing_line_color="#dc2626",
    ),
    row=1,
    col=1,
)
fig_1h.add_trace(
    go.Bar(
        x=plot_1h.index,
        y=plot_1h["volume"],
        name="Volume",
        marker_color=volume_colors_1h,
        hovertemplate="Volume: %{y:,.0f}<extra></extra>",
    ),
    row=2,
    col=1,
)

fig_1h.update_layout(
    title=dict(text=f"ADANIGREEN 1H Price — Donchian {DONCHIAN_PERIOD} Channel", x=0.01),
    height=850,
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=70, r=30, t=100, b=50),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    xaxis_rangeslider_visible=False,
)
# Hide weekends and overnight gaps so hourly candles sit next to each other.
fig_1h.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
        dict(bounds=[15.5, 9.25], pattern="hour"),
    ]
)
fig_1h.update_yaxes(title_text="Price (₹)", row=1, col=1)
fig_1h.update_yaxes(title_text="Volume", row=2, col=1)
fig_1h.show()

## Entry, stop loss and target

**Buy** when the 4H `ema_50_angle` is above **+55°** and an hourly close breaks the upper Donchian 20 band.

- entry = hourly close (LTP)
- stop loss = `ATR_MULTIPLE` × ATR(14) below the entry (set `SL_METHOD = "swing"` for the original lowest low of the last 10 hourly candles)
- target = entry + 2 × (entry − stop loss)

**Sell** when the 4H `ema_50_angle` is below **−55°** and an hourly close breaks the lower Donchian 20 band.

- entry = hourly close (LTP)
- stop loss = `ATR_MULTIPLE` × ATR(14) above the entry (set `SL_METHOD = "swing"` for the original highest high of the last 10 hourly candles)
- target = entry − 2 × (stop loss − entry)

Two details keep the signals honest:

- The band compared against is the **previous** candle's band, so a candle cannot break a channel it helped build.
- The 4H angle attached to each hourly candle comes from the **last completed** 4H candle, so no signal uses a number that was not yet known.

In [10]:
ANGLE_THRESHOLD = 55
SL_LOOKBACK = 10
TARGET_MULTIPLE = 2
ATR_PERIOD = 14
ATR_MULTIPLE = 2.0
SL_METHOD = "swing"  # "swing" for the 10-candle extreme (the rule as specified), "atr" for an ATR band

signals = intraday.copy()

# Attach the last COMPLETED 4H angle to every hourly candle (no look-ahead).
angle_4h = price[ANGLE_COL].shift(1).rename("ema_50_angle_4h").dropna()
signals = pd.merge_asof(
    signals.sort_index(),
    angle_4h.to_frame().sort_index(),
    left_index=True,
    right_index=True,
    direction="backward",
)

# Break the PREVIOUS candle's channel, not the one this candle helped build.
prev_upper = signals["donchian_upper"].shift(1)
prev_lower = signals["donchian_lower"].shift(1)
signals["prev_donchian_upper"] = prev_upper
signals["prev_donchian_lower"] = prev_lower

breaks_upper = signals["close"] > prev_upper
breaks_lower = signals["close"] < prev_lower

long_raw = breaks_upper & (signals["ema_50_angle_4h"] > ANGLE_THRESHOLD)
short_raw = breaks_lower & (signals["ema_50_angle_4h"] < -ANGLE_THRESHOLD)

# Keep only the first candle of each run so one breakout is one signal.
signals["long_signal"] = long_raw & ~long_raw.shift(1, fill_value=False)
signals["short_signal"] = short_raw & ~short_raw.shift(1, fill_value=False)

signals["side"] = pd.Series(pd.NA, index=signals.index, dtype="string")
signals.loc[signals["long_signal"], "side"] = "Buy"
signals.loc[signals["short_signal"], "side"] = "Sell"

# Two stop inputs: the 10-candle extreme, and Wilder's ATR for a volatility band.
signals["swing_low"] = signals["low"].rolling(SL_LOOKBACK, min_periods=SL_LOOKBACK).min()
signals["swing_high"] = signals["high"].rolling(SL_LOOKBACK, min_periods=SL_LOOKBACK).max()

prev_close = signals["close"].shift(1)
true_range = pd.concat(
    [
        signals["high"] - signals["low"],
        (signals["high"] - prev_close).abs(),
        (signals["low"] - prev_close).abs(),
    ],
    axis=1,
).max(axis=1)
signals["atr"] = true_range.ewm(alpha=1 / ATR_PERIOD, adjust=False, min_periods=ATR_PERIOD).mean()


def stop_levels(frame: pd.DataFrame, method: str = SL_METHOD, atr_mult: float = ATR_MULTIPLE) -> pd.Series:
    """Stop price for each signal candle, either the swing extreme or an ATR band."""
    if method == "swing":
        long_stop, short_stop = frame["swing_low"], frame["swing_high"]
    elif method == "atr":
        band = atr_mult * frame["atr"]
        long_stop, short_stop = frame["close"] - band, frame["close"] + band
    else:
        raise ValueError(f"Unknown stop method: {method!r}")
    return pd.Series(
        np.where(frame["long_signal"], long_stop, np.where(frame["short_signal"], short_stop, np.nan)),
        index=frame.index,
    )


def apply_stops(frame: pd.DataFrame, method: str = SL_METHOD, atr_mult: float = ATR_MULTIPLE) -> pd.DataFrame:
    """Attach entry, stop loss and a TARGET_MULTIPLE-of-risk target to every signal."""
    out = frame.copy()
    out["entry"] = out["close"].where(out["side"].notna())
    out["stop_loss"] = stop_levels(out, method, atr_mult)
    risk = (out["entry"] - out["stop_loss"]).abs()
    out["risk"] = risk
    out["target"] = np.where(
        out["long_signal"],
        out["entry"] + TARGET_MULTIPLE * risk,
        np.where(out["short_signal"], out["entry"] - TARGET_MULTIPLE * risk, np.nan),
    )
    out["risk_pct"] = (risk / out["entry"] * 100).where(out["side"].notna())
    return out


signals = apply_stops(signals)

print(
    f"Stop: {SL_METHOD} "
    + (f"({ATR_MULTIPLE:g} x ATR {ATR_PERIOD})" if SL_METHOD == "atr" else f"({SL_LOOKBACK}-candle extreme)")
    + f" | median risk {signals['risk_pct'].median():.2f}% of entry"
)

print(
    f"{int(signals['long_signal'].sum())} buy and "
    f"{int(signals['short_signal'].sum())} sell signal(s) "
    f"between {signals.index.min():%d %b %Y} and {signals.index.max():%d %b %Y}."
)

Stop: swing (10-candle extreme) | median risk 4.96% of entry
55 buy and 81 sell signal(s) between 01 Dec 2023 and 19 Aug 2026.


### Signal list

In [11]:
signal_table = (
    signals.loc[
        signals["side"].notna(),
        [
            "side",
            "close",
            "ema_50_angle_4h",
            "prev_donchian_upper",
            "prev_donchian_lower",
            "entry",
            "stop_loss",
            "target",
            "risk",
            "risk_pct",
        ],
    ]
    .rename_axis("timestamp")
    .reset_index()
)

display(signal_table)

,timestamp,side,close,ema_50_angle_4h,prev_donchian_upper,prev_donchian_lower,entry,stop_loss,target,risk,risk_pct
0,2024-01-10 10:15:00+05:30,Buy,"1,739.90",81.72,"1,725.00","1,660.00","1,739.90","1,670.00","1,879.70",69.90,4.02
1,2024-01-29 09:15:00+05:30,Buy,"1,726.65",63.16,"1,695.00","1,629.00","1,726.65","1,629.00","1,921.95",97.65,5.66
2,2024-02-07 09:15:00+05:30,Buy,"1,917.05",69.90,"1,744.95","1,651.60","1,917.05","1,653.00","2,445.15",264.05,13.77
3,2024-02-15 09:15:00+05:30,Buy,"1,921.95",79.35,"1,908.00","1,734.30","1,921.95","1,782.40","2,201.05",139.55,7.26
4,2024-02-19 11:15:00+05:30,Buy,"1,960.85",81.58,"1,959.70","1,837.00","1,960.85","1,891.65","2,099.25",69.20,3.53
...,...,...,...,...,...,...,...,...,...,...,...
131,2026-07-23 13:15:00+05:30,Sell,"1,376.50",-66.60,"1,558.70","1,388.00","1,376.50","1,527.80","1,073.90",151.30,10.99
132,2026-07-24 09:15:00+05:30,Sell,"1,369.30",-75.87,"1,558.70","1,372.00","1,369.30","1,508.50","1,090.90",139.20,10.17
133,2026-08-10 09:15:00+05:30,Sell,"1,361.30",-68.76,"1,398.00","1,365.70","1,361.30","1,389.90","1,304.10",28.60,2.10
134,2026-08-12 09:15:00+05:30,Sell,"1,334.70",-67.02,"1,416.00","1,354.00","1,334.70","1,416.00","1,172.10",81.30,6.09


### 1H chart with signals

Green triangles mark buy signals, red triangles mark sell signals. Each signal draws a red stop-loss line and a green target line running forward from the entry candle.

In [12]:
# Use None for the full history, or set a number such as 500 for recent candles only.
PLOT_BARS_SIGNALS = 500
LEVEL_BARS = 20  # how far the stop-loss and target lines run forward

plot_sig = signals.copy() if PLOT_BARS_SIGNALS is None else signals.tail(PLOT_BARS_SIGNALS).copy()
window_signals = plot_sig.loc[plot_sig["side"].notna()]
buys = window_signals.loc[window_signals["side"] == "Buy"]
sells = window_signals.loc[window_signals["side"] == "Sell"]
marker_pad = max(float((plot_sig["high"] - plot_sig["low"]).median()) * 0.8, 0.5)


def level_segments(frame: pd.DataFrame, column: str) -> tuple[list, list]:
    """Build a broken-line trace that draws one horizontal level per signal."""
    xs: list = []
    ys: list = []
    positions = plot_sig.index
    for timestamp, row in frame.iterrows():
        start = positions.get_loc(timestamp)
        end = min(start + LEVEL_BARS, len(positions) - 1)
        xs.extend([positions[start], positions[end], None])
        ys.extend([row[column], row[column], None])
    return xs, ys


sl_x, sl_y = level_segments(window_signals, "stop_loss")
tgt_x, tgt_y = level_segments(window_signals, "target")

volume_colors_sig = plot_sig["close"].ge(plot_sig["open"]).map(
    {True: "rgba(22, 163, 74, 0.42)", False: "rgba(220, 38, 38, 0.42)"}
)

fig_sig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.62, 0.20, 0.18],
    vertical_spacing=0.04,
)

fig_sig.add_trace(
    go.Scatter(
        x=plot_sig.index,
        y=plot_sig["donchian_upper"],
        mode="lines",
        name=f"Donchian {DONCHIAN_PERIOD} upper",
        line=dict(color="#0ea5e9", width=1.4),
        hovertemplate="Upper: ₹%{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)
fig_sig.add_trace(
    go.Scatter(
        x=plot_sig.index,
        y=plot_sig["donchian_lower"],
        mode="lines",
        name=f"Donchian {DONCHIAN_PERIOD} lower",
        line=dict(color="#0ea5e9", width=1.4),
        fill="tonexty",
        fillcolor="rgba(14, 165, 233, 0.12)",
        hovertemplate="Lower: ₹%{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)
fig_sig.add_trace(
    go.Candlestick(
        x=plot_sig.index,
        open=plot_sig["open"],
        high=plot_sig["high"],
        low=plot_sig["low"],
        close=plot_sig["close"],
        name="ADANIGREEN 1H",
        increasing_line_color="#16a34a",
        decreasing_line_color="#dc2626",
    ),
    row=1,
    col=1,
)
fig_sig.add_trace(
    go.Scatter(
        x=sl_x,
        y=sl_y,
        mode="lines",
        name="Stop loss",
        line=dict(color="#b91c1c", width=1.4, dash="dot"),
        hoverinfo="skip",
    ),
    row=1,
    col=1,
)
fig_sig.add_trace(
    go.Scatter(
        x=tgt_x,
        y=tgt_y,
        mode="lines",
        name="Target",
        line=dict(color="#15803d", width=1.4, dash="dot"),
        hoverinfo="skip",
    ),
    row=1,
    col=1,
)
fig_sig.add_trace(
    go.Scatter(
        x=buys.index,
        y=buys["low"] - marker_pad,
        mode="markers",
        name="Buy signal",
        customdata=buys[["entry", "stop_loss", "target", "ema_50_angle_4h"]],
        marker=dict(symbol="triangle-up", color="#15803d", size=13, line=dict(color="white", width=1)),
        hovertemplate=(
            "%{x|%d %b %Y %H:%M}<br>Buy"
            "<br>Entry: ₹%{customdata[0]:,.2f}"
            "<br>SL: ₹%{customdata[1]:,.2f}"
            "<br>Target: ₹%{customdata[2]:,.2f}"
            "<br>4H EMA angle: %{customdata[3]:.2f}°<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)
fig_sig.add_trace(
    go.Scatter(
        x=sells.index,
        y=sells["high"] + marker_pad,
        mode="markers",
        name="Sell signal",
        customdata=sells[["entry", "stop_loss", "target", "ema_50_angle_4h"]],
        marker=dict(symbol="triangle-down", color="#b91c1c", size=13, line=dict(color="white", width=1)),
        hovertemplate=(
            "%{x|%d %b %Y %H:%M}<br>Sell"
            "<br>Entry: ₹%{customdata[0]:,.2f}"
            "<br>SL: ₹%{customdata[1]:,.2f}"
            "<br>Target: ₹%{customdata[2]:,.2f}"
            "<br>4H EMA angle: %{customdata[3]:.2f}°<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)
fig_sig.add_trace(
    go.Scatter(
        x=plot_sig.index,
        y=plot_sig["ema_50_angle_4h"],
        mode="lines",
        name="4H EMA 50 angle",
        line=dict(color="#2563eb", width=1.5),
        hovertemplate="4H EMA 50 angle: %{y:.2f}°<extra></extra>",
    ),
    row=2,
    col=1,
)
for level, color in ((ANGLE_THRESHOLD, "#15803d"), (-ANGLE_THRESHOLD, "#b91c1c")):
    fig_sig.add_hline(y=level, line=dict(color=color, width=1, dash="dash"), row=2, col=1)
fig_sig.add_trace(
    go.Bar(
        x=plot_sig.index,
        y=plot_sig["volume"],
        name="Volume",
        marker_color=volume_colors_sig,
        hovertemplate="Volume: %{y:,.0f}<extra></extra>",
    ),
    row=3,
    col=1,
)

fig_sig.update_layout(
    title=dict(
        text=(
            f"ADANIGREEN 1H — Donchian {DONCHIAN_PERIOD} breakouts with "
            f"4H EMA {EMA_PERIOD} angle beyond ±{ANGLE_THRESHOLD}°"
        ),
        x=0.01,
    ),
    height=1000,
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=70, r=30, t=100, b=50),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    xaxis_rangeslider_visible=False,
)
fig_sig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
        dict(bounds=[15.5, 9.25], pattern="hour"),
    ]
)
fig_sig.update_yaxes(title_text="Price (₹)", row=1, col=1)
fig_sig.update_yaxes(title_text="Angle (°)", row=2, col=1)
fig_sig.update_yaxes(title_text="Volume", row=3, col=1)
fig_sig.show()

print(f"{len(buys)} buy and {len(sells)} sell signal(s) inside the plotted window.")

15 buy and 5 sell signal(s) inside the plotted window.


## Backtest

The same rules run through [`backtesting`](https://kernc.github.io/backtesting.py/) on the hourly candles.

- A signal fills at the **close of the signal candle** (`trade_on_close=True`), which is the LTP the rules refer to.
- Stop loss and target ride along as bracket orders, so whichever the candle touches first closes the trade.
- **One position at a time.** While a trade is open every new signal is ignored, including an opposite one — the trade leaves only at its own stop loss or target.
- Warm-up candles are dropped, so the test starts once both the 4H EMA 50 and the Donchian 20 have full windows.

In [13]:
from backtesting import Backtest, Strategy

BT_COLUMNS = ["long_signal", "short_signal", "stop_loss", "target"]

def to_bt_frame(frame: pd.DataFrame) -> pd.DataFrame:
    """Shape a signal frame the way `backtesting` wants it, warm-up candles dropped."""
    out = frame.rename(
        columns={"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"}
    )[["Open", "High", "Low", "Close", "Volume", "ema_50_angle_4h", *BT_COLUMNS]].copy()
    # A signal row needs its levels; every other row only needs valid OHLC, an angle and a band.
    ready = out["ema_50_angle_4h"].notna() & frame["donchian_upper"].notna() & frame["atr"].notna()
    out = out.loc[ready].copy()
    out[["long_signal", "short_signal"]] = out[["long_signal", "short_signal"]].astype(float)
    out[["stop_loss", "target"]] = out[["stop_loss", "target"]].fillna(0.0)
    return out


bt_data = to_bt_frame(signals)

print(f"Backtest window: {bt_data.index.min():%d %b %Y} to {bt_data.index.max():%d %b %Y} ({len(bt_data):,} candles)")


class DonchianEmaAngle(Strategy):
    """Donchian 20 breakout on 1H, gated by the 4H EMA 50 angle, with a 2R target."""

    def init(self):
        # Columns are read inside next(); an array captured here stays full length,
        # so its [-1] would always point at the final candle of the whole test.
        pass

    def next(self):
        # A trade is already running, so every signal is ignored until it closes.
        if self.position:
            return

        price = self.data.Close[-1]
        sl = float(self.data.stop_loss[-1])
        tp = float(self.data.target[-1])

        if self.data.long_signal[-1] > 0.5:
            # The broker rejects a bracket that does not straddle the fill price.
            if sl < price < tp:
                self.buy(sl=sl, tp=tp)
        elif self.data.short_signal[-1] > 0.5:
            if tp < price < sl:
                self.sell(sl=sl, tp=tp)

/home/amit/anaconda3/envs/visualizer/lib/python3.11/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

Backtest window: 09 Jan 2024 to 19 Aug 2026 (4,512 candles)


### Run the backtest

In [14]:
INITIAL_CASH = 1_000_000
COMMISSION = 0.001  # 0.1% per side, a rough stand-in for brokerage plus slippage

bt = Backtest(
    bt_data,
    DonchianEmaAngle,
    cash=INITIAL_CASH,
    commission=COMMISSION,
    trade_on_close=True,
    finalize_trades=True,
)

stats = bt.run()
display(stats.drop(index=[k for k in stats.index if k.startswith("_")]).to_frame("value"))

,value
Start,2024-01-09 13:15:00+05:30
End,2026-08-19 15:15:00+05:30
Duration,953 days 02:00:00
Exposure Time [%],58.84
Equity Final [$],"612,734.20"
Equity Peak [$],"1,106,355.75"
Commissions [$],"54,003.95"
Return [%],-38.73
Buy & Hold Return [%],-23.59
Return (Ann.) [%],-17.37


### Trade list

In [15]:
trades = stats["_trades"].copy()
trades["Side"] = np.where(trades["Size"] > 0, "Buy", "Sell")
trades["ReturnPct"] = trades["ReturnPct"] * 100

trade_table = trades[
    ["Side", "EntryTime", "ExitTime", "EntryPrice", "ExitPrice", "Size", "PnL", "ReturnPct", "Duration"]
].reset_index(drop=True)

wins = trades.loc[trades["PnL"] > 0]
losses = trades.loc[trades["PnL"] <= 0]
taken = len(trades)
available = int(bt_data["long_signal"].sum() + bt_data["short_signal"].sum())
print(
    f"{taken} trades taken out of {available} signals "
    f"({available - taken} skipped while a trade was open)"
)
print(
    f"{len(wins)} winners, {len(losses)} losers | "
    f"avg win ₹{wins['PnL'].mean():,.0f} vs avg loss ₹{losses['PnL'].mean():,.0f}"
)
display(trade_table)

40 trades taken out of 136 signals (96 skipped while a trade was open)
14 winners, 26 losers | avg win ₹70,459 vs avg loss ₹-52,834


,Side,EntryTime,ExitTime,EntryPrice,ExitPrice,Size,PnL,ReturnPct,Duration
0,Buy,2024-01-10 10:15:00+05:30,2024-01-16 13:15:00+05:30,"1,739.90","1,670.00",574,"-42,079.88",-4.21,6 days 03:00:00
1,Buy,2024-01-29 09:15:00+05:30,2024-02-07 09:15:00+05:30,"1,726.65","1,921.95",554,"106,174.88",11.10,9 days 00:00:00
2,Buy,2024-02-07 09:15:00+05:30,2024-03-13 09:15:00+05:30,"1,917.05","1,653.00",554,"-148,261.51",-13.96,35 days 00:00:00
3,Sell,2024-04-18 14:15:00+05:30,2024-05-16 11:15:00+05:30,"1,762.45","1,849.90",-519,"-47,261.36",-5.17,27 days 21:00:00
4,Buy,2024-05-31 13:15:00+05:30,2024-06-04 09:15:00+05:30,"1,961.65","1,840.10",442,"-55,405.47",-6.39,3 days 20:00:00
5,Sell,2024-06-26 15:15:00+05:30,2024-06-27 14:15:00+05:30,"1,770.00","1,809.85",-458,"-19,890.87",-2.45,0 days 23:00:00
6,Sell,2024-07-05 13:15:00+05:30,2024-07-09 09:15:00+05:30,"1,754.85","1,775.00",-451,"-10,679.61",-1.35,3 days 20:00:00
7,Sell,2024-07-19 14:15:00+05:30,2024-07-23 09:15:00+05:30,"1,724.30","1,759.00",-453,"-17,297.03",-2.21,3 days 19:00:00
8,Buy,2024-08-19 14:15:00+05:30,2024-09-13 14:15:00+05:30,"1,901.20","1,787.00",402,"-47,391.06",-6.20,25 days 00:00:00
9,Sell,2024-09-13 14:15:00+05:30,2024-09-16 09:15:00+05:30,"1,785.80","1,850.00",-401,"-27,202.16",-3.80,2 days 19:00:00


### Equity curve and trade P&L

In [16]:
equity = stats["_equity_curve"]

fig_bt = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.62, 0.38],
    vertical_spacing=0.06,
    subplot_titles=("Equity curve", "Drawdown"),
)

fig_bt.add_trace(
    go.Scatter(
        x=equity.index,
        y=equity["Equity"],
        mode="lines",
        name="Equity",
        line=dict(color="#2563eb", width=1.8),
        hovertemplate="Equity: ₹%{y:,.0f}<extra></extra>",
    ),
    row=1,
    col=1,
)
fig_bt.add_hline(y=INITIAL_CASH, line=dict(color="#94a3b8", width=1, dash="dot"), row=1, col=1)
fig_bt.add_trace(
    go.Scatter(
        x=equity.index,
        y=equity["DrawdownPct"] * -100,
        mode="lines",
        name="Drawdown",
        line=dict(color="#dc2626", width=1.4),
        fill="tozeroy",
        fillcolor="rgba(220, 38, 38, 0.18)",
        hovertemplate="Drawdown: %{y:.2f}%<extra></extra>",
    ),
    row=2,
    col=1,
)

fig_bt.update_layout(
    title=dict(text="Donchian 20 breakout + 4H EMA 50 angle — backtest equity", x=0.01),
    height=720,
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=80, r=30, t=110, b=50),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_bt.update_yaxes(title_text="Equity (₹)", row=1, col=1)
fig_bt.update_yaxes(title_text="Drawdown (%)", row=2, col=1)
fig_bt.show()

fig_pnl = go.Figure(
    go.Bar(
        x=trades["ExitTime"],
        y=trades["PnL"],
        marker_color=np.where(trades["PnL"] > 0, "#16a34a", "#dc2626"),
        hovertemplate="%{x|%d %b %Y %H:%M}<br>P&L: ₹%{y:,.0f}<extra></extra>",
        name="Trade P&L",
    )
)
fig_pnl.update_layout(
    title=dict(text="Profit and loss per trade", x=0.01),
    height=420,
    template="plotly_white",
    margin=dict(l=80, r=30, t=80, b=50),
    yaxis_title="P&L (₹)",
    showlegend=False,
)
fig_pnl.show()

Run `bt.plot(open_browser=False)` in a fresh cell for the interactive Bokeh report that ships with `backtesting`, if you want its trade-by-trade view on top of these charts.

### Stop comparison

The same signals, the same one-trade-at-a-time rule, only the stop changes. A wider stop means fewer stop-outs but a target that sits further away, so the two pull against each other.

In [17]:
def run_variant(label: str, method: str, atr_mult: float = ATR_MULTIPLE) -> dict:
    """Re-price the stops, rerun the backtest and pull out the headline numbers."""
    variant = to_bt_frame(apply_stops(signals, method, atr_mult))
    result = Backtest(
        variant,
        DonchianEmaAngle,
        cash=INITIAL_CASH,
        commission=COMMISSION,
        trade_on_close=True,
        finalize_trades=True,
    ).run()
    return {
        "stop": label,
        "trades": int(result["# Trades"]),
        "win_rate": result["Win Rate [%]"],
        "return_pct": result["Return [%]"],
        "max_dd_pct": result["Max. Drawdown [%]"],
        "profit_factor": result["Profit Factor"],
        "expectancy_pct": result["Expectancy [%]"],
        "avg_duration": result["Avg. Trade Duration"],
    }


variants = [(f"swing {SL_LOOKBACK}", "swing", ATR_MULTIPLE)]
variants += [(f"{m:g} x ATR", "atr", m) for m in (1.5, 2.0, 2.5, 3.0, 4.0, 5.0)]

comparison = pd.DataFrame([run_variant(*v) for v in variants])
display(comparison)

,stop,trades,win_rate,return_pct,max_dd_pct,profit_factor,expectancy_pct,avg_duration
0,swing 10,40,35.00,-38.73,-62.80,0.87,-0.62,14 days 02:00:00
1,1.5 x ATR,86,34.88,-24.99,-33.29,0.82,-0.29,2 days 00:00:00
2,2 x ATR,74,33.78,-13.73,-36.58,0.95,-0.09,3 days 16:00:00
3,2.5 x ATR,60,31.67,-29.27,-42.61,0.85,-0.40,6 days 06:00:00
4,3 x ATR,57,31.58,-39.21,-59.20,0.81,-0.62,7 days 00:00:00
5,4 x ATR,45,33.33,-17.97,-48.44,0.98,-0.08,10 days 15:00:00
6,5 x ATR,30,33.33,-12.29,-55.61,1.04,0.20,20 days 06:00:00
